# BÀI TẬP THỰC HÀNH: PROMPT ENGINEERING

## Mục tiêu
- Hiểu và áp dụng cấu trúc một Prompt chuẩn
- Thực hành các kỹ thuật Prompt Engineering phổ biến
- So sánh hiệu quả của các kỹ thuật khác nhau

---

## Chuẩn bị
Trong bài tập này, chúng ta sẽ sử dụng OpenAI API hoặc bất kỳ LLM nào bạn có quyền truy cập.


In [ ]:
# Cài đặt thư viện cần thiết
# pip install openai python-dotenv

import os
from openai import OpenAI
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Khởi tạo client
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Hàm tiện ích để gọi LLM
def call_llm(prompt, model="gpt-3.5-turbo", temperature=0.7, max_tokens=500):
    """
    Gọi OpenAI API với prompt đã cho
    """
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            temperature=temperature,
            max_tokens=max_tokens
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Lỗi: {str(e)}"

print("✓ Setup hoàn tất!")

---

## Phần 1: CẤU TRÚC PROMPT CHUẨN

### Lý thuyết
Một prompt hiệu quả thường có cấu trúc:
1. **Role/Context** (Vai trò/Ngữ cảnh): Xác định vai trò của AI
2. **Task** (Nhiệm vụ): Mô tả rõ ràng công việc cần làm
3. **Format** (Định dạng): Chỉ định cách trình bày kết quả
4. **Constraints** (Ràng buộc): Giới hạn hoặc yêu cầu đặc biệt
5. **Examples** (Ví dụ): Cung cấp mẫu (nếu cần)

### 🎯 BÀI TẬP 1.1: Xây dựng Prompt Chuẩn

**Tình huống thực tế:** Bạn đang xây dựng chatbot tư vấn sức khỏe cho một phòng khám nha khoa.

**Yêu cầu:** Viết prompt với đầy đủ 4 thành phần (Role, Task, Format, Constraints) để AI tư vấn cho khách hàng về vấn đề "đau răng khôn".

In [ ]:
# BÀI TẬP 1.1: Viết prompt của bạn ở đây

# ❌ Prompt kém hiệu quả (để so sánh)
bad_prompt = "Tôi bị đau răng khôn. Phải làm sao?"

# ✅ Prompt chuẩn - HÃY HOÀN THIỆN
good_prompt = """
[Role/Context]: Bạn là chuyên gia tư vấn nha khoa với 10 năm kinh nghiệm...

[Task]: ...

[Format]: ...

[Constraints]: ...
"""

# Test prompt
print("=" * 50)
print("KẾT QUẢ VỚI PROMPT KÉM:")
print("=" * 50)
result_bad = call_llm(bad_prompt)
print(result_bad)

print("\n" + "=" * 50)
print("KẾT QUẢ VỚI PROMPT CHUẨN:")
print("=" * 50)
result_good = call_llm(good_prompt)
print(result_good)

# TODO: So sánh 2 kết quả và rút ra nhận xét

### 🎯 BÀI TẬP 1.2: Tối ưu hóa Prompt

**Tình huống:** Bạn cần trích xuất thông tin từ email khách hàng để nhập vào CRM.

**Cho trước:** Email mẫu dưới đây

**Yêu cầu:** Viết prompt để trích xuất: Tên, Email, Số điện thoại, Vấn đề, Mức độ khẩn cấp

In [ ]:
# BÀI TẬP 1.2: Trích xuất thông tin từ email

sample_email = """
Chào anh/chị,

Em là Nguyễn Minh Anh, đang gặp vấn đề nghiêm trọng với sản phẩm máy giặt 
LG model WF-123 mua hôm 15/12/2025. Máy bị kêu to và rung lắc quá mức, 
ảnh hưởng đến hàng xóm. Em cần được hỗ trợ gấp trong ngày hôm nay.

Thông tin liên hệ:
- SĐT: 0912345678
- Email: minhanh.nguyen@email.com
- Địa chỉ: 123 Nguyễn Huệ, Quận 1, TP.HCM

Mong nhận được phản hồi sớm!
Trân trọng,
Minh Anh
"""

# TODO: Viết prompt để trích xuất thông tin
extraction_prompt = f"""
[Viết prompt của bạn ở đây]

Email:
{sample_email}
"""

# Test prompt
result = call_llm(extraction_prompt, temperature=0)
print(result)

# Mục tiêu: Kết quả trả về dạng JSON có thể parse được
# {
#   "ten": "Nguyễn Minh Anh",
#   "email": "minhanh.nguyen@email.com",
#   "sdt": "0912345678",
#   "van_de": "Máy giặt kêu to và rung lắc",
#   "muc_do_khan_cap": "Cao"
# }

---

## Phần 2: ZERO-SHOT vs FEW-SHOT LEARNING

### Lý thuyết
- **Zero-shot**: AI thực hiện nhiệm vụ không có ví dụ mẫu
- **Few-shot**: Cung cấp vài ví dụ để AI học pattern

### 🎯 BÀI TẬP 2.1: Phân loại Sentiment (Cảm xúc)

**Tình huống:** Phân tích review sản phẩm trên e-commerce

**Yêu cầu:** 
1. Thử Zero-shot
2. Thử Few-shot với 3 ví dụ
3. So sánh độ chính xác

In [ ]:
# BÀI TẬP 2.1: Zero-shot vs Few-shot

# Dữ liệu test
test_reviews = [
    "Sản phẩm tạm ổn, giá hơi cao so với chất lượng",
    "Tuyệt vời! Đóng gói cẩn thận, ship nhanh. Sẽ ủng hộ shop lâu dài",
    "Thất vọng! Hàng không đúng mô tả, shop không phản hồi tin nhắn",
    "Bình thường, không có gì đặc biệt",
    "Chất lượng vượt mong đợi với mức giá này"
]

# ========== ZERO-SHOT ==========
zero_shot_prompt = """
Phân loại cảm xúc của review sau thành: POSITIVE, NEGATIVE, hoặc NEUTRAL.
Chỉ trả về một từ.

Review: {review}
Sentiment:
"""

print("=" * 50)
print("ZERO-SHOT LEARNING")
print("=" * 50)
for review in test_reviews:
    prompt = zero_shot_prompt.format(review=review)
    result = call_llm(prompt, temperature=0)
    print(f"Review: {review[:50]}...")
    print(f"Sentiment: {result}\n")

# ========== FEW-SHOT ==========
# TODO: Viết few-shot prompt với ít nhất 3 ví dụ
few_shot_prompt = """
Phân loại cảm xúc của review. Ví dụ:

Review: "Shop phục vụ nhiệt tình, hàng đẹp lắm ạ!"
Sentiment: POSITIVE

Review: "Hàng dở tệ, giao chậm, không khuyến khích mua"
Sentiment: NEGATIVE

Review: "Cũng được, không tốt lắm nhưng chấp nhận được"
Sentiment: NEUTRAL

[Thêm ví dụ của bạn...]

Bây giờ phân loại:
Review: {review}
Sentiment:
"""

print("=" * 50)
print("FEW-SHOT LEARNING")
print("=" * 50)
for review in test_reviews:
    prompt = few_shot_prompt.format(review=review)
    result = call_llm(prompt, temperature=0)
    print(f"Review: {review[:50]}...")
    print(f"Sentiment: {result}\n")

# TODO: Phân tích kết quả và rút ra nhận xét

### 🎯 BÀI TẬP 2.2: Few-shot cho Task phức tạp

**Tình huống:** Chuyển đổi mô tả sản phẩm từ tiếng Việt sang copy quảng cáo hấp dẫn

**Yêu cầu:** Tạo Few-shot prompt với phong cách marketing chuyên nghiệp

In [ ]:
# BÀI TẬP 2.2: Few-shot Marketing Copy

# Dữ liệu mẫu
product_descriptions = [
    {
        "input": "Áo thun cotton 100%, có 5 màu, giá 150k",
        "expected": "Tự tin tỏa sáng với áo thun cotton 100% siêu mềm mại! 🌟 5 màu trendy cho mọi phong cách. Chỉ 150k - Đáng đồng tiền bát gạo!"
    },
    {
        "input": "Tai nghe bluetooth, pin 20h, chống ồn",
        "expected": "🎧 Trải nghiệm âm thanh đỉnh cao với tai nghe bluetooth! Pin 20h bất tận + Chống ồn thông minh. Đắm chìm vào thế giới của riêng bạn!"
    }
]

# TODO: Xây dựng few-shot prompt
marketing_prompt = """
Chuyển mô tả sản phẩm thành copy quảng cáo hấp dẫn, ngắn gọn, có emoji phù hợp.

[Thêm ví dụ từ product_descriptions ở đây...]

Bây giờ viết copy cho:
Mô tả: {description}
Copy:
"""

# Test với sản phẩm mới
test_products = [
    "Bình giữ nhiệt 500ml, giữ nóng 12h, giữ lạnh 24h, inox 304",
    "Sách 'Tư duy ngược' 300 trang, tác giả nổi tiếng, kiến thức thực tế",
    "Đèn ngủ thông minh, điều khiển bằng giọng nói, 16 triệu màu"
]

for product in test_products:
    # TODO: Hoàn thiện prompt
    result = call_llm(marketing_prompt.format(description=product))
    print(f"Sản phẩm: {product}")
    print(f"Copy: {result}\n")
    print("-" * 50)

---

## Phần 3: CHAIN-OF-THOUGHT (CoT) - TƯ DUY LOGIC TỪNG BƯỚC

### Lý thuyết
Chain-of-Thought yêu cầu AI giải thích từng bước suy luận trước khi đưa ra kết luận. Đặc biệt hiệu quả với:
- Bài toán logic
- Phân tích phức tạp
- Ra quyết định có nhiều yếu tố

### 🎯 BÀI TẬP 3.1: Giải bài toán Logic

**Tình huống:** Hệ thống tự động tính toán chiết khấu phức tạp

In [ ]:
# BÀI TẬP 3.1: Bài toán tính chiết khấu

problem = """
Một cửa hàng có chính sách giảm giá như sau:
- Giảm 10% cho đơn hàng từ 500k
- Giảm thêm 5% nếu là khách hàng VIP
- Giảm thêm 50k nếu thanh toán online
- Tối đa giảm 30% tổng giá trị đơn hàng

Khách hàng A mua hàng trị giá 800k, là VIP, thanh toán online.
Hỏi khách hàng phải trả bao nhiêu?
"""

# ❌ Prompt thông thường (không CoT)
normal_prompt = f"""
{problem}

Trả lời ngắn gọn:
"""

print("=" * 50)
print("KHÔNG SỬ DỤNG CoT")
print("=" * 50)
result_normal = call_llm(normal_prompt, temperature=0)
print(result_normal)

# ✅ Prompt với Chain-of-Thought
cot_prompt = f"""
{problem}

Hãy giải quyết từng bước:
1. Xác định các chính sách áp dụng
2. Tính toán từng loại giảm giá
3. Kiểm tra giới hạn giảm giá tối đa
4. Tính số tiền cuối cùng

Trình bày chi tiết:
"""

print("\n" + "=" * 50)
print("SỬ DỤNG CHAIN-OF-THOUGHT")
print("=" * 50)
result_cot = call_llm(cot_prompt, temperature=0, max_tokens=800)
print(result_cot)

# So sánh độ chính xác và logic của 2 kết quả

### 🎯 BÀI TẬP 3.2: Phân tích và Ra quyết định

**Tình huống thực tế:** Hệ thống tự động đề xuất sản phẩm dựa trên nhiều tiêu chí

**Yêu cầu:** Sử dụng CoT để phân tích và đưa ra đề xuất hợp lý

In [ ]:
# BÀI TẬP 3.2: Đề xuất sản phẩm với CoT

customer_profile = """
Thông tin khách hàng:
- Độ tuổi: 28
- Nghề nghiệp: Lập trình viên
- Sở thích: Công nghệ, đọc sách, thể thao
- Ngân sách: 5-7 triệu đồng
- Mục đích: Quà tặng sinh nhật bạn gái (25 tuổi, làm marketing, thích du lịch)

Danh sách sản phẩm:
1. Máy ảnh Fujifilm X-T30 (giá: 15 triệu) - chất lượng tuyệt vời nhưng vượt ngân sách
2. Túi xách Michael Kors (giá: 6 triệu) - thương hiệu nổi tiếng, phù hợp công sở
3. Máy đọc sách Kindle (giá: 3 triệu) - tiện lợi nhưng ít lãng mạn
4. Vòng tay Pandora (giá: 5.5 triệu) - đẹp, ý nghĩa, trong ngân sách
5. Loa bluetooth JBL Charge 5 (giá: 4 triệu) - chất lượng tốt nhưng ít phù hợp
"""

# TODO: Viết CoT prompt để phân tích và đề xuất
cot_recommendation_prompt = f"""
{customer_profile}

Nhiệm vụ: Đề xuất sản phẩm phù hợp nhất.

Phân tích từng bước:
1. Xác định các tiêu chí quan trọng (ngân sách, sở thích người nhận, tính lãng mạn...)
2. Đánh giá từng sản phẩm theo các tiêu chí
3. Loại trừ các lựa chọn không phù hợp
4. So sánh các lựa chọn còn lại
5. Đưa ra đề xuất cuối cùng với lý do

Hãy phân tích:
"""

result = call_llm(cot_recommendation_prompt, temperature=0.3, max_tokens=1000)
print(result)

# TODO: Thử thay đổi thông tin khách hàng và test lại

### 🎯 BÀI TẬP 3.3: Debugging Code với CoT

**Challenge:** Sử dụng CoT để phân tích lỗi code

In [ ]:
# BÀI TẬP 3.3: Debug với CoT

buggy_code = """
def calculate_average(numbers):
    total = 0
    for num in numbers:
        total += num
    return total / len(numbers)

# Test
scores = [85, 90, 78, 92, 88]
print(calculate_average(scores))

# Bug: Khi gọi với list rỗng sẽ bị lỗi ZeroDivisionError
print(calculate_average([]))
"""

# TODO: Viết CoT prompt để phân tích và sửa lỗi
debug_prompt = f"""
Phân tích và sửa lỗi code sau theo từng bước:

```python
{buggy_code}
```

Quy trình phân tích:
1. Đọc và hiểu code đang làm gì
2. Xác định các trường hợp test (bình thường và edge cases)
3. Tìm ra lỗi hoặc điểm yếu
4. Giải thích tại sao lỗi xảy ra
5. Đề xuất cách sửa
6. Viết code đã fix

Hãy phân tích:
"""

result = call_llm(debug_prompt, temperature=0.2, max_tokens=1000)
print(result)

---

## Phần 4: ReAct (REASON + ACT) - KẾT HỢP SUY LUẬN VÀ HÀNH ĐỘNG

### Lý thuyết
ReAct kết hợp:
- **Reasoning**: Suy luận về vấn đề
- **Acting**: Thực hiện hành động cụ thể
- **Observing**: Quan sát kết quả và điều chỉnh

Phù hợp với: Agent AI, Task planning, Interactive problem solving

### 🎯 BÀI TẬP 4.1: Travel Planning Agent

**Tình huống:** Xây dựng agent lên kế hoạch du lịch

In [ ]:
# BÀI TẬP 4.1: ReAct cho Travel Planning

# Giả lập các công cụ mà agent có thể sử dụng
def search_flights(origin, destination, date):
    """Giả lập tìm kiếm chuyến bay"""
    return f"Tìm thấy 3 chuyến bay từ {origin} đến {destination} vào {date}: VN123 (2tr), VJ456 (1.5tr), QH789 (1.8tr)"

def search_hotels(location, checkin, nights):
    """Giả lập tìm kiếm khách sạn"""
    return f"Tìm thấy khách sạn tại {location}: Hotel A (800k/đêm), Hotel B (1.2tr/đêm), Hotel C (600k/đêm)"

def search_attractions(location):
    """Giả lập tìm địa điểm tham quan"""
    return f"Địa điểm nổi tiếng ở {location}: Chùa, Bảo tàng, Biển, Chợ đêm, Công viên"

# Yêu cầu của người dùng
user_request = """
Tôi muốn đi du lịch Đà Nẵng 3 ngày 2 đêm, khởi hành từ Hà Nội vào 15/02/2026.
Ngân sách khoảng 7 triệu. Tôi thích biển và đồ ăn hải sản.
"""

# TODO: Viết ReAct prompt
react_prompt = f"""
Bạn là travel planning agent. Sử dụng quy trình ReAct để lên kế hoạch du lịch.

Công cụ có sẵn:
- search_flights(origin, destination, date): Tìm chuyến bay
- search_hotels(location, checkin, nights): Tìm khách sạn
- search_attractions(location): Tìm địa điểm tham quan

Quy trình ReAct:
1. THOUGHT (Suy nghĩ): Phân tích yêu cầu, xác định bước tiếp theo
2. ACTION (Hành động): Quyết định dùng tool nào với params gì
3. OBSERVATION (Quan sát): Nhận kết quả từ tool
4. Lặp lại cho đến khi hoàn thành

Yêu cầu:
{user_request}

Bắt đầu:
THOUGHT 1: [Phân tích yêu cầu...]
"""

# Simulate ReAct cycle
print("=" * 60)
print("REACT AGENT ĐANG LÀM VIỆC...")
print("=" * 60)

result = call_llm(react_prompt, temperature=0.3, max_tokens=1500)
print(result)

# TODO: Mở rộng bài tập bằng cách thực sự gọi các hàm và cập nhật context

### 🎯 BÀI TẬP 4.2: Customer Support Agent với ReAct

**Tình huống:** Agent tự động xử lý yêu cầu khách hàng

In [ ]:
# BÀI TẬP 4.2: ReAct Customer Support Agent

# Giả lập database và tools
def check_order_status(order_id):
    """Kiểm tra trạng thái đơn hàng"""
    orders = {
        "DH001": "Đang giao - Dự kiến 16h chiều nay",
        "DH002": "Đã giao - 14/01/2026",
        "DH003": "Đang xử lý - Chờ xác nhận thanh toán"
    }
    return orders.get(order_id, "Không tìm thấy đơn hàng")

def check_product_stock(product_id):
    """Kiểm tra tồn kho"""
    stock = {
        "SP001": 150,
        "SP002": 0,
        "SP003": 25
    }
    return f"Còn {stock.get(product_id, 0)} sản phẩm trong kho"

def create_return_request(order_id, reason):
    """Tạo yêu cầu trả hàng"""
    return f"Đã tạo mã trả hàng TH-{order_id} với lý do: {reason}"

def check_refund_policy():
    """Chính sách hoàn tiền"""
    return "Hoàn tiền trong 7 ngày nếu sản phẩm lỗi. 15 ngày nếu không vừa ý."

# Các tình huống khách hàng
customer_issues = [
    {
        "issue": "Tôi đặt đơn hàng DH001 hôm qua nhưng chưa nhận được. Bao giờ tới?",
        "context": "Khách hàng lo lắng về đơn hàng"
    },
    {
        "issue": "Sản phẩm SP002 có còn hàng không? Tôi muốn mua 10 cái",
        "context": "Khách hàng muốn mua sản phẩm"
    },
    {
        "issue": "Đơn DH002 tôi nhận được bị lỗi. Muốn trả hàng và hoàn tiền",
        "context": "Khách hàng không hài lòng, muốn trả hàng"
    }
]

# TODO: Viết ReAct agent xử lý từng tình huống
react_support_template = """
Bạn là customer support agent. Sử dụng ReAct để xử lý yêu cầu khách hàng.

Available Tools:
- check_order_status(order_id): Tra cứu đơn hàng
- check_product_stock(product_id): Kiểm tra tồn kho
- create_return_request(order_id, reason): Tạo đơn trả hàng
- check_refund_policy(): Xem chính sách hoàn tiền

Customer Issue: {issue}

ReAct Process:
THOUGHT: [Phân tích vấn đề, xác định cần làm gì]
ACTION: [Tool cần dùng với parameters]
OBSERVATION: [Kết quả giả định từ tool]
... (lặp lại nếu cần)
FINAL RESPONSE: [Trả lời khách hàng một cách thân thiện, chuyên nghiệp]

Start:
"""

print("=" * 70)
print("CUSTOMER SUPPORT AGENT DEMO")
print("=" * 70)

for idx, case in enumerate(customer_issues, 1):
    print(f"\n{'='*70}")
    print(f"CASE {idx}: {case['context']}")
    print(f"{'='*70}")
    print(f"Customer: {case['issue']}\n")
    
    prompt = react_support_template.format(issue=case['issue'])
    response = call_llm(prompt, temperature=0.3, max_tokens=1000)
    print(response)
    print("\n")

### 🎯 BÀI TẬP 4.3: Research Agent (Nâng cao)

**Challenge:** Xây dựng agent tự động research và tổng hợp thông tin

**Yêu cầu:** Agent cần tự động:
1. Phân tích câu hỏi
2. Xác định cần tìm kiếm gì
3. Giả lập tìm kiếm (search)
4. Tổng hợp và trả lời

In [ ]:
# BÀI TẬP 4.3: ReAct Research Agent (Advanced)

def web_search(query):
    """Giả lập tìm kiếm web"""
    # Trong thực tế sẽ gọi API như Google Search, Bing, etc.
    mock_results = {
        "python langchain": "LangChain là framework để xây dựng LLM applications. Hỗ trợ chains, agents, memory...",
        "prompt engineering best practices": "Best practices: Be specific, provide examples, use clear structure, iterate and test...",
        "vietnam gdp 2025": "GDP Việt Nam 2025 ước đạt 7.2%, đứng đầu ASEAN..."
    }
    for key in mock_results:
        if key in query.lower():
            return mock_results[key]
    return f"Kết quả tìm kiếm cho '{query}': [Nội dung giả lập]"

def wikipedia_lookup(topic):
    """Giả lập tra Wikipedia"""
    return f"Wikipedia về '{topic}': [Thông tin tổng quan giả lập]"

def calculator(expression):
    """Tính toán"""
    try:
        return f"Kết quả: {eval(expression)}"
    except:
        return "Lỗi tính toán"

# Research questions
research_questions = [
    "So sánh LangChain và LlamaIndex cho việc xây dựng RAG system",
    "Tính GDP bình quân đầu người của Việt Nam năm 2025 (dân số 100 triệu)",
    "Các best practices cho prompt engineering trong production system"
]

# TODO: Viết ReAct research agent
react_research_template = """
Bạn là research agent chuyên nghiệp. Sử dụng ReAct để trả lời câu hỏi.

Available Tools:
- web_search(query): Tìm kiếm thông tin trên web
- wikipedia_lookup(topic): Tra cứu Wikipedia
- calculator(expression): Tính toán

Question: {question}

ReAct Chain:
THOUGHT 1: [Phân tích câu hỏi, xác định chiến lược research]
ACTION 1: [Tool + parameters]
OBSERVATION 1: [Kết quả]

THOUGHT 2: [Đánh giá kết quả, quyết định bước tiếp theo]
ACTION 2: [Tool + parameters nếu cần]
OBSERVATION 2: [Kết quả]

... (tiếp tục nếu cần)

FINAL THOUGHT: [Tổng hợp tất cả thông tin]
ANSWER: [Câu trả lời cuối cùng, đầy đủ và chính xác]

Begin:
"""

print("=" * 70)
print("RESEARCH AGENT DEMO")
print("=" * 70)

for question in research_questions:
    print(f"\n{'='*70}")
    print(f"QUESTION: {question}")
    print(f"{'='*70}\n")
    
    prompt = react_research_template.format(question=question)
    response = call_llm(prompt, temperature=0.4, max_tokens=1500)
    print(response)
    print("\n")

# TODO: Thử với câu hỏi của riêng bạn

---

## Phần 5: BÀI TẬP TỔNG HỢP (MINI PROJECT)

### 🎯 PROJECT: XÂY DỰNG CHATBOT TƯ VẤN SẢN PHẨM

**Yêu cầu:** Kết hợp tất cả các kỹ thuật đã học để xây dựng chatbot thông minh

**Tính năng:**
1. Hiểu ngữ cảnh và mục đích khách hàng (Few-shot)
2. Phân tích nhu cầu logic (CoT)
3. Tự động tìm kiếm và đề xuất sản phẩm (ReAct)
4. Trả lời chuyên nghiệp với cấu trúc prompt chuẩn

**Tiêu chí đánh giá:**
- ✅ Áp dụng đúng cấu trúc prompt
- ✅ Sử dụng Few-shot cho classification
- ✅ Áp dụng CoT cho reasoning
- ✅ Implement ReAct cho decision making

In [ ]:
# MINI PROJECT: Smart Product Advisor Chatbot

# Database sản phẩm
product_database = [
    {"id": "LT001", "name": "Laptop Dell XPS 13", "price": 25000000, "category": "laptop", 
     "specs": "i7, 16GB RAM, 512GB SSD", "rating": 4.5},
    {"id": "LT002", "name": "Laptop Macbook Air M2", "price": 28000000, "category": "laptop",
     "specs": "M2, 8GB RAM, 256GB SSD", "rating": 4.8},
    {"id": "LT003", "name": "Laptop Asus Gaming", "price": 22000000, "category": "laptop",
     "specs": "i5, 16GB RAM, 512GB SSD, RTX 3050", "rating": 4.3},
    {"id": "PH001", "name": "iPhone 15 Pro", "price": 29000000, "category": "phone",
     "specs": "A17 Pro, 128GB", "rating": 4.7},
    {"id": "PH002", "name": "Samsung Galaxy S24", "price": 20000000, "category": "phone",
     "specs": "Snapdragon 8 Gen 3, 256GB", "rating": 4.6},
]

# TODO: Implement các function xử lý
def classify_intent(message):
    """
    Phân loại ý định khách hàng: product_search, price_inquiry, comparison, etc.
    Sử dụng Few-shot Learning
    """
    # TODO: Implement với few-shot prompt
    pass

def analyze_requirements(message):
    """
    Phân tích yêu cầu chi tiết của khách hàng
    Sử dụng Chain-of-Thought
    """
    # TODO: Implement với CoT prompt
    pass

def search_and_recommend(requirements):
    """
    Tìm kiếm và đề xuất sản phẩm phù hợp
    Sử dụng ReAct
    """
    # TODO: Implement với ReAct prompt
    pass

def generate_response(intent, analysis, recommendations):
    """
    Tạo câu trả lời cuối cùng với cấu trúc chuẩn
    """
    # TODO: Implement với structured prompt
    pass

# Main chatbot flow
def chatbot(user_message):
    """
    Main function xử lý toàn bộ luồng
    """
    print(f"🧑 User: {user_message}\n")
    
    # Step 1: Classify intent
    print("⚙️  Đang phân tích ý định...")
    intent = classify_intent(user_message)
    print(f"   → Intent: {intent}\n")
    
    # Step 2: Analyze requirements
    print("⚙️  Đang phân tích yêu cầu...")
    analysis = analyze_requirements(user_message)
    print(f"   → Analysis: {analysis}\n")
    
    # Step 3: Search & recommend
    print("⚙️  Đang tìm kiếm sản phẩm phù hợp...")
    recommendations = search_and_recommend(analysis)
    print(f"   → Found: {recommendations}\n")
    
    # Step 4: Generate response
    print("⚙️  Đang tạo câu trả lời...")
    response = generate_response(intent, analysis, recommendations)
    
    print(f"🤖 Bot: {response}\n")
    print("=" * 70 + "\n")

# Test cases
test_conversations = [
    "Tôi cần một laptop để code và học AI, ngân sách khoảng 25 triệu",
    "So sánh iPhone 15 Pro và Samsung S24 cho tôi",
    "Có laptop gaming nào dưới 23 triệu không?",
    "Tôi muốn mua điện thoại chụp ảnh đẹp, pin trâu, giá khoảng 20 triệu"
]

# TODO: Implement đầy đủ các function và test
print("=" * 70)
print("SMART PRODUCT ADVISOR CHATBOT")
print("=" * 70 + "\n")

for message in test_conversations:
    chatbot(message)

# TODO: 
# 1. Hoàn thiện các function với prompt engineering techniques
# 2. Test với nhiều cases khác nhau
# 3. Cải thiện và tối ưu prompts
# 4. Thêm error handling
# 5. Đánh giá performance

---

## 📝 ĐÁNH GIÁ VÀ TỰ KIỂM TRA

### Checklist hoàn thành bài tập:

#### Phần 1: Cấu trúc Prompt Chuẩn
- [ ] Hoàn thành Bài tập 1.1: Viết prompt với đầy đủ Role, Task, Format, Constraints
- [ ] Hoàn thành Bài tập 1.2: Trích xuất thông tin từ email
- [ ] So sánh kết quả giữa prompt tốt và kém

#### Phần 2: Zero-shot & Few-shot
- [ ] Hoàn thành Bài tập 2.1: Phân loại sentiment
- [ ] Hoàn thành Bài tập 2.2: Marketing copy generation
- [ ] Quan sát sự khác biệt giữa Zero-shot và Few-shot

#### Phần 3: Chain-of-Thought
- [ ] Hoàn thành Bài tập 3.1: Bài toán tính chiết khấu
- [ ] Hoàn thành Bài tập 3.2: Đề xuất sản phẩm
- [ ] Hoàn thành Bài tập 3.3: Debug code
- [ ] Thấy được lợi ích của CoT trong reasoning tasks

#### Phần 4: ReAct
- [ ] Hoàn thành Bài tập 4.1: Travel planning agent
- [ ] Hoàn thành Bài tập 4.2: Customer support agent
- [ ] Hoàn thành Bài tập 4.3: Research agent
- [ ] Hiểu cách kết hợp Thought-Action-Observation

#### Phần 5: Mini Project
- [ ] Implement chatbot với đầy đủ các kỹ thuật
- [ ] Test với nhiều test cases
- [ ] Tối ưu prompts để cải thiện kết quả

---

## 🎓 CÂU HỎI ÔN TẬP

1. **Khi nào nên dùng Few-shot thay vì Zero-shot?**
2. **Chain-of-Thought phù hợp với loại bài toán nào?**
3. **Sự khác biệt giữa CoT và ReAct là gì?**
4. **Làm thế nào để đánh giá một prompt có hiệu quả?**
5. **Các lỗi thường gặp khi viết prompt là gì?**

---

## 💡 GỢI Ý MỞ RỘNG

### Nâng cao bài tập:
1. **Multi-language Support**: Thử các prompt với tiếng Anh và so sánh
2. **A/B Testing**: So sánh nhiều phiên bản prompt khác nhau
3. **Real API Integration**: Tích hợp với API thật (Google Search, Database)
4. **Evaluation Metrics**: Xây dựng metrics để đo lường hiệu quả
5. **Prompt Templates Library**: Tạo thư viện các prompt templates tái sử dụng

### Real-world Applications:
- Customer service automation
- Content generation pipeline
- Data extraction from documents
- Code generation & debugging assistant
- Research & analysis automation

---

## 📚 TÀI LIỆU THAM KHẢO

- [Prompt Engineering Guide](https://www.promptingguide.ai/)
- [OpenAI Best Practices](https://platform.openai.com/docs/guides/prompt-engineering)
- [Chain-of-Thought Prompting](https://arxiv.org/abs/2201.11903)
- [ReAct Paper](https://arxiv.org/abs/2210.03629)
- [LangChain Documentation](https://python.langchain.com/)

---

**Chúc bạn học tập hiệu quả! 🚀**